[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradypuz/TorchCode-web-ui/blob/master/templates/43_grpo_complete_loss.ipynb)

# 🔴 Hard: GRPO Loss - Complete

Implement an **original-style Group Relative Policy Optimization (GRPO) loss** with group-relative rewards, PPO-style clipping, a reference-policy KL penalty, and response-token masking.

This is the next step after **GRPO Loss**. Here, `B` prompts each have `G` sampled responses, with up to `T` response tokens per response. The responses were sampled from the rollout (old) policy. All three log-probability tensors describe **the same sampled tokens at the same positions**, evaluated by different policies. They are already selected token log-probabilities, not vocabulary logits.

### Inputs and output

| Input | Shape | Meaning |
|---|---|---|
| `policy_logps` | `(B, G, T)` | Current policy log-probabilities; this is the training graph |
| `old_logps` | `(B, G, T)` | Fixed rollout-policy log-probabilities used in the importance ratio |
| `ref_logps` | `(B, G, T)` | Fixed reference-policy log-probabilities used in the KL penalty |
| `rewards` | `(B, G)` | One scalar reward per sampled response |
| `token_mask` | `(B, G, T)` | Boolean or 0/1 mask: valid response token = 1; padding/prompt token = 0 |
| `beta`, `epsilon` | scalars | KL weight and PPO clipping width; defaults `0.04` and `0.2` |

Return one scalar loss tensor. Treat `old_logps`, `ref_logps`, `rewards`, and `token_mask` as constants; gradients must reach only valid entries of `policy_logps`.

### Objective

1. Normalize rewards **within each prompt's group**, using population standard deviation:

$$
A_{b,g}=\operatorname{stop\_grad}\left(\frac{r_{b,g}-\mu_b}{\max(\sigma_b,10^{-6})}\right).
$$

`mean` and `std` reduce over `G` (dimension 1), and `std` uses `unbiased=False`. Broadcast each response's advantage across its token dimension. Equal-reward groups, including `G = 1`, have zero advantages.

2. Compute a ratio for each valid token:

$$
\rho_{b,g,t}=\exp\left(\ell^{\text{policy}}_{b,g,t}-\operatorname{stop\_grad}(\ell^{\text{old}}_{b,g,t})\right).
$$

3. Form the clipped policy loss:

$$
P_{b,g,t}=-\min\left(\rho_{b,g,t}A_{b,g},\;\operatorname{clip}(\rho_{b,g,t},1-\epsilon,1+\epsilon)A_{b,g}\right).
$$

4. Add the nonnegative **sampled-token KL estimator** against the reference policy:

$$
d_{b,g,t}=\operatorname{stop\_grad}(\ell^{\text{ref}}_{b,g,t})-\ell^{\text{policy}}_{b,g,t},\qquad
K_{b,g,t}=\operatorname{expm1}(d_{b,g,t})-d_{b,g,t}.
$$

`expm1(d)` computes `exp(d) - 1` more accurately near zero. This is a penalty estimated from sampled tokens, not an exact sum over the whole vocabulary.

5. Average valid token losses **within each response**, then average across all `B * G` responses:

$$
L=\frac{1}{BG}\sum_{b,g}\frac{\sum_t m_{b,g,t}\left(P_{b,g,t}+\beta K_{b,g,t}\right)}{\max\left(1,\sum_t m_{b,g,t}\right)}.
$$

This gives responses equal weight even when their lengths differ; do not use one global mean over all valid tokens. A fully masked response contributes zero but remains included in the final `B * G` denominator.

### Requirements

- Assume `B >= 1`, `G >= 1`, and `T >= 1`. Floating-point inputs have matching dtype and device; the mask is on the same device. Assume `beta >= 0` and `0 <= epsilon < 1`.
- Rewards and valid-token log-probabilities are finite, and their exponential ratios are representable in the input dtype.
- **Padding entries may contain arbitrary values, including NaN or Inf. Replace masked log-probabilities with zero before exponentiation.** Multiplying an infinite intermediate by a zero mask afterward is insufficient.
- Use the population standard deviation clamped to `1e-6`; do not add the PPO clipping `epsilon` to the standard deviation.
- Keep the gradient graph through current policy log-probabilities. Detach the other inputs.

### Signature

```python
from torch import Tensor

def grpo_complete_loss(
    policy_logps: Tensor,
    old_logps: Tensor,
    ref_logps: Tensor,
    rewards: Tensor,
    token_mask: Tensor,
    beta: float = 0.04,
    epsilon: float = 0.2,
) -> Tensor:
    """Clipped GRPO loss with a reference-policy KL penalty and response masking.

    policy_logps: (B, G, T) current log-probs of the sampled response tokens
    old_logps: (B, G, T) rollout-policy log-probs of those same tokens
    ref_logps: (B, G, T) reference-policy log-probs of those same tokens
    rewards: (B, G) scalar rewards for G responses to each of B prompts
    token_mask: (B, G, T) bool or 0/1 mask of valid response tokens
    returns: scalar loss; gradients flow only through valid policy_logps
    """
```

“Complete” refers to this fuller **loss function**, not an entire GRPO trainer. Sampling, computing token log-probabilities/rewards, optimizer updates, and distributed training are outside the exercise. Modern GRPO variants may use different normalization or KL choices; implement the exact objective above.


In [ ]:
# Install the source version that includes this new exercise in Colab.
# No-op in JupyterLab/Docker, where the repository package is already available.
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q git+https://github.com/bradypuz/TorchCode-web-ui.git@master')
except ImportError:
    pass


In [ ]:
import torch
from torch import Tensor


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def grpo_complete_loss(
    policy_logps: Tensor,
    old_logps: Tensor,
    ref_logps: Tensor,
    rewards: Tensor,
    token_mask: Tensor,
    beta: float = 0.04,
    epsilon: float = 0.2,
) -> Tensor:
    """Clipped GRPO loss with a reference-policy KL penalty and response masking.

    policy_logps: (B, G, T) current log-probs of the sampled response tokens
    old_logps: (B, G, T) rollout-policy log-probs of those same tokens
    ref_logps: (B, G, T) reference-policy log-probs of those same tokens
    rewards: (B, G) scalar rewards for G responses to each of B prompts
    token_mask: (B, G, T) bool or 0/1 mask of valid response tokens
    returns: scalar loss; gradients flow only through valid policy_logps
    """
    pass  # normalize rewards, clip policy ratios, add KL, mask and reduce


In [ ]:
# 🧪 Debug
old_logps = torch.full((1, 2, 2), -2.0)
ratios = torch.tensor([[[1.4, 1.0], [0.6, 1.0]]])
policy_logps = (old_logps + ratios.log()).requires_grad_()
ref_logps = old_logps.clone()
rewards = torch.tensor([[1.0, -1.0]])
token_mask = torch.tensor([[[True, True], [True, False]]])
loss = grpo_complete_loss(policy_logps, old_logps, ref_logps, rewards, token_mask)
print('Loss:', loss.item())
# Expected: approximately -0.146376 with beta=0.04 and epsilon=0.2


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_complete_loss')
